# Feather BERTs — causal check: do the predictive heads actually drive HANS behaviour?

**Session options:** Accelerator **GPU T4** or **P100**, Internet **On**. No inputs needed.

Run cells 1–6 by hand and check the first model's line in cell 6 looks sensible, then **Save Version → Save & Run All (Commit)**. About 1 hour. Download `causal_out.zip` from the Output tab.

In [ ]:
%%writefile feather_eval.py
#!/usr/bin/env python3
"""
Steps 2-3 of the "mechanistic vs representational predictors of OOD behaviour" project.

Population: the 100 BERT-base models McCoy et al. (2020) fine-tuned on MNLI, hosted as
    connectivity/feather_berts_0 ... connectivity/feather_berts_99   (Juneja et al. 2023)

Subcommands
-----------
  check    Step 2. Load a few models; verify (a) the label order, (b) that our forward pass
           reproduces the logits stored in each repo's mnli_eval_bert_{i}.json, and (c) that
           MNLI matched-dev accuracy falls in McCoy et al.'s reported 83.6-84.8% range.
  eval     Step 3a. Run every model on MNLI matched-dev and all 30k HANS items and save
           per-example logits. Resumable: models with an existing output file are skipped.
  analyze  Step 3b. Is the OOD outcome real? Split-half reliability and between-model spread
           of HANS lexical-overlap non-entailment accuracy, plus diagnostics. Prints PASS/FAIL
           against pre-set gates.

Typical use (on a GPU machine with Hugging Face access)
-------------------------------------------------------
  python feather_eval.py check   --models 0 1 2 3 4
  python feather_eval.py eval    --models all --purge
  python feather_eval.py analyze

Label conventions (verified against the repos' own eval files):
  model output index : 0 = contradiction, 1 = entailment, 2 = neutral   (Google BERT MnliProcessor)
  GLUE/HF MNLI label : 0 = entailment,    1 = neutral,    2 = contradiction
  HANS               : prediction is "entailment" iff argmax == 1; neutral + contradiction
                       collapse to "non-entailment" (as in McCoy et al.).
"""

import argparse
import csv
import json
import os
import platform
import sys
import time
import urllib.request
from pathlib import Path

import numpy as np

# ----------------------------------------------------------------------------- constants

REPO_TMPL = "connectivity/feather_berts_{i}"
N_MODELS = 100
MODEL_LABELS = {"contradiction": 0, "entailment": 1, "neutral": 2}
ENT = MODEL_LABELS["entailment"]
GLUE_TO_MODEL = np.array([1, 2, 0])  # GLUE ent/neu/con -> model index
HANS_URL = "https://raw.githubusercontent.com/tommccoy1/hans/master/heuristics_evaluation_set.txt"
MCCOY_MNLI_RANGE = (0.836, 0.848)  # reported MNLI matched-dev range across the 100 models
MAX_LEN = 128                       # max_seq_length used by McCoy et al. (Google BERT default)

# Pre-registered gates for step 3 (set before seeing data; do not tune).
GATE_SPLIT_HALF_R = 0.80   # raw (not Spearman-Brown) stratified split-half correlation
GATE_SD_RATIO = 3.0        # between-model SD / binomial sampling SD
GATE_RANGE = 0.10          # max - min accuracy across models


# ----------------------------------------------------------------------------- data

def load_hans(path):
    import pandas as pd
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Downloading HANS to {path} ...")
        urllib.request.urlretrieve(HANS_URL, path)
    d = pd.read_csv(path, sep="\t", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
    assert len(d) == 30000, f"HANS should have 30000 rows, got {len(d)}"
    return {
        "premise": d["sentence1"].tolist(),
        "hypothesis": d["sentence2"].tolist(),
        "gold_ent": (d["gold_label"] == "entailment").to_numpy(),
        "heuristic": d["heuristic"].to_numpy(),
        "subcase": d["subcase"].to_numpy(),
        "template": d["template"].to_numpy(),
        "pairID": d["pairID"].to_numpy(),
    }


def load_mnli(tsv=None):
    """MNLI matched dev. Labels returned in MODEL index space."""
    if tsv:  # GLUE dev_matched.tsv
        prem, hyp, lab = [], [], []
        name2model = MODEL_LABELS
        with open(tsv, newline="", encoding="utf-8") as f:
            r = csv.DictReader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
            for row in r:
                if row["gold_label"] not in name2model:
                    continue
                prem.append(row["sentence1"]); hyp.append(row["sentence2"])
                lab.append(name2model[row["gold_label"]])
        return {"premise": prem, "hypothesis": hyp, "label": np.array(lab)}
    from datasets import load_dataset
    ds = load_dataset("nyu-mll/glue", "mnli", split="validation_matched")
    lab = np.array(ds["label"])
    keep = lab >= 0
    return {
        "premise": [p for p, k in zip(ds["premise"], keep) if k],
        "hypothesis": [h for h, k in zip(ds["hypothesis"], keep) if k],
        "label": GLUE_TO_MODEL[lab[keep]],
    }


# ----------------------------------------------------------------------------- models

def model_source(i, args):
    """Return (local_dir, revision_sha). Downloads from the Hub unless --local_tmpl is set."""
    if args.local_tmpl:
        return Path(args.local_tmpl.format(i=i)), "local"
    from huggingface_hub import snapshot_download
    repo = REPO_TMPL.format(i=i)
    path = snapshot_download(
        repo,
        allow_patterns=["config.json", "pytorch_model.bin", "*.safetensors", "tokenizer*",
                        "vocab.txt", "special_tokens_map.json"],
    )
    return Path(path), Path(path).name  # snapshot dir name is the commit sha


def purge_from_cache(i):
    from huggingface_hub import scan_cache_dir
    repo = REPO_TMPL.format(i=i)
    info = scan_cache_dir()
    for r in info.repos:
        if r.repo_id == repo:
            info.delete_revisions(*[rev.commit_hash for rev in r.revisions]).execute()


def load_model(path, device):
    import torch
    from transformers import AutoTokenizer, BertForSequenceClassification
    tok = AutoTokenizer.from_pretrained(path)
    model = BertForSequenceClassification.from_pretrained(path).to(device).eval()
    # Sanity: uncased tokenizer and 3-way head.
    if tok("Hello World")["input_ids"] != tok("hello world")["input_ids"]:
        print("  WARNING: tokenizer is not lower-casing; McCoy models are bert-base-uncased.")
    assert model.config.num_labels == 3, model.config.num_labels
    return tok, model


def run_model(tok, model, premises, hypotheses, device, batch_size=256):
    """Logits (float32, original order). Batches are formed over length-sorted inputs."""
    import torch
    n = len(premises)
    order = np.argsort([len(p) + len(h) for p, h in zip(premises, hypotheses)])
    out = np.empty((n, 3), dtype=np.float32)
    with torch.inference_mode():
        for s in range(0, n, batch_size):
            idx = order[s:s + batch_size]
            enc = tok([premises[j] for j in idx], [hypotheses[j] for j in idx],
                      truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            out[idx] = model(**enc).logits.float().cpu().numpy()
    return out


def pick_device(arg):
    import torch
    if arg != "auto":
        return arg
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def parse_models(spec):
    if spec == ["all"]:
        return list(range(N_MODELS))
    return [int(x) for x in spec]


# ----------------------------------------------------------------------------- step 2: check

def read_stored_eval(path_dir, i, args):
    """The repo's own mnli_eval_bert_{i}.json: label_dict + samples with logits and label."""
    fname = f"mnli_eval_bert_{i}.json"
    if args.local_tmpl:
        p = Path(path_dir) / fname
    else:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(REPO_TMPL.format(i=i), fname)
    with open(p) as f:
        d = json.load(f)
    return d["label_dict"], d["samples"]


def split_stored_sentence(s):
    s = s.strip()
    if s.startswith("[CLS]"):
        s = s[len("[CLS]"):]
    parts = [p.strip() for p in s.split("[SEP]")]
    parts = [p for p in parts if p]
    if len(parts) != 2:
        return None
    return parts[0], parts[1]


def cmd_check(args):
    import torch
    device = pick_device(args.device)
    print(f"device={device}  torch={torch.__version__}")
    mnli = load_mnli(args.mnli_tsv)
    print(f"MNLI matched dev: {len(mnli['label'])} labelled examples")
    rows, all_ok = [], True
    for i in parse_models(args.models):
        t0 = time.time()
        path, sha = model_source(i, args)
        tok, model = load_model(path, device)

        # (a) label order
        label_dict, samples = read_stored_eval(path, i, args)
        label_ok = {k.lower(): int(v) for k, v in label_dict.items()} == MODEL_LABELS

        # (b) reproduce stored logits on the first N stored examples
        pairs, stored = [], []
        for smp in samples[: args.n_reproduce]:
            pr = split_stored_sentence(smp["sentence"])
            if pr is not None:
                pairs.append(pr); stored.append(smp["logits"])
        stored = np.array(stored, dtype=np.float32)
        ours = run_model(tok, model, [p for p, _ in pairs], [h for _, h in pairs], device,
                         args.batch_size)
        absdiff = np.abs(ours - stored)
        argmax_agree = float((ours.argmax(1) == stored.argmax(1)).mean())

        # accuracy implied by the stored file itself (all samples)
        st_logits = np.array([s["logits"] for s in samples], dtype=np.float32)
        st_labels = np.array([s["label"] for s in samples])
        stored_acc = float((st_logits.argmax(1) == st_labels).mean())

        # (c) our accuracy on MNLI matched dev with our label mapping
        logits = run_model(tok, model, mnli["premise"], mnli["hypothesis"], device, args.batch_size)
        acc = float((logits.argmax(1) == mnli["label"]).mean())

        lo, hi = MCCOY_MNLI_RANGE
        ok = (label_ok and argmax_agree >= 0.995 and float(np.median(absdiff)) < 1e-2
              and lo - 0.005 <= acc <= hi + 0.005 and abs(acc - stored_acc) < 0.005)
        all_ok &= ok
        rows.append(dict(model=i, sha=sha, label_order_ok=label_ok, n_stored=len(samples),
                         n_reproduced=len(pairs), median_absdiff=float(np.median(absdiff)),
                         max_absdiff=float(absdiff.max()), argmax_agree=argmax_agree,
                         stored_file_acc=stored_acc, our_mnli_acc=acc,
                         seconds=round(time.time() - t0, 1), PASS=ok))
        r = rows[-1]
        print(f"[model {i:>2}] label_order={'ok' if label_ok else 'MISMATCH'}  "
              f"reproduce: argmax_agree={argmax_agree:.4f} median|dlogit|={r['median_absdiff']:.2e} "
              f"max|dlogit|={r['max_absdiff']:.2e}  stored_acc={stored_acc:.4f}  "
              f"our_acc={acc:.4f}  ({r['seconds']}s)  {'PASS' if ok else 'FAIL'}")
        del model
        if device == "cuda":
            torch.cuda.empty_cache()

    out = Path(args.out_dir); out.mkdir(parents=True, exist_ok=True)
    with open(out / "step2_check.json", "w") as f:
        json.dump({"rows": rows, "env": env_info(), "all_pass": all_ok}, f, indent=2)
    print("\nSTEP 2:", "PASS - population is usable; run `eval`." if all_ok else
          "FAIL - inspect step2_check.json before continuing.")
    print("Interpretation: our_acc ~= stored_file_acc and argmax_agree ~= 1 means weights, "
          "tokenisation and label mapping all match the original evaluation.")


# ----------------------------------------------------------------------------- step 3a: eval

def cmd_eval(args):
    import torch
    device = pick_device(args.device)
    out = Path(args.out_dir); (out / "logits").mkdir(parents=True, exist_ok=True)
    mnli = load_mnli(args.mnli_tsv)
    hans = load_hans(args.hans_path)
    # Save aligned metadata once so `analyze` never depends on loader order at analysis time.
    np.savez(out / "meta.npz", mnli_label=mnli["label"], hans_gold_ent=hans["gold_ent"],
             hans_heuristic=hans["heuristic"], hans_subcase=hans["subcase"],
             hans_template=hans["template"], hans_pairID=hans["pairID"])
    models = parse_models(args.models)
    print(f"device={device}  models={len(models)}  MNLI n={len(mnli['label'])}  HANS n=30000")
    failed = {}
    for k, i in enumerate(models):
        f = out / "logits" / f"model_{i:03d}.npz"
        if f.exists():
            print(f"[{k+1}/{len(models)}] model {i}: exists, skipping"); continue
        t0 = time.time()
        try:
            path, sha = model_source(i, args)
            tok, model = load_model(path, device)
            ml = run_model(tok, model, mnli["premise"], mnli["hypothesis"], device, args.batch_size)
            hl = run_model(tok, model, hans["premise"], hans["hypothesis"], device, args.batch_size)
        except Exception as e:  # one bad repo must not kill a 100-model run; rerun later to retry
            failed[i] = f"{type(e).__name__}: {e}"
            print(f"[{k+1}/{len(models)}] model {i}: FAILED ({failed[i][:200]})")
            continue
        tmp = f.with_suffix(".tmp.npz")
        np.savez(tmp, mnli_logits=ml, hans_logits=hl, sha=sha)
        os.replace(tmp, f)  # atomic: a crash never leaves a half-written file
        mnli_acc = (ml.argmax(1) == mnli["label"]).mean()
        lo = (hans["heuristic"] == "lexical_overlap") & ~hans["gold_ent"]
        lo_acc = (hl[lo].argmax(1) != ENT).mean()
        print(f"[{k+1}/{len(models)}] model {i}: MNLI={mnli_acc:.4f}  HANS-LO-nonent={lo_acc:.4f}  "
              f"({time.time()-t0:.0f}s)")
        del model
        if device == "cuda":
            torch.cuda.empty_cache()
        if args.purge and not args.local_tmpl:
            purge_from_cache(i)
    with open(out / "eval_env.json", "w") as fh:
        json.dump({**env_info(), "failed": failed}, fh, indent=2)
    done = len(list((out / "logits").glob("model_*.npz")))
    print(f"\nDone: {done} models have logits. Failed this run: {sorted(failed) or 'none'}"
          + ("  (rerun the same command to retry; finished models are skipped)" if failed else ""))


# ----------------------------------------------------------------------------- step 3b: analyze

def bimodality_coefficient(x):
    from scipy.stats import kurtosis, skew
    n = len(x)
    g, k = skew(x, bias=False), kurtosis(x, bias=False)
    return (g ** 2 + 1) / (k + 3 * (n - 1) ** 2 / ((n - 2) * (n - 3)))


def cmd_analyze(args):
    import itertools
    import pandas as pd
    from scipy.stats import pearsonr, spearmanr

    out = Path(args.out_dir)
    meta = np.load(out / "meta.npz", allow_pickle=True)
    files = sorted((out / "logits").glob("model_*.npz"))
    if len(files) < 10:
        sys.exit(f"Only {len(files)} models evaluated; run `eval` first.")
    ids = [int(f.stem.split("_")[1]) for f in files]
    gold_ent, heur, sub = meta["hans_gold_ent"], meta["hans_heuristic"], meta["hans_subcase"]

    # correctness matrices: models x items
    mnli_correct, hans_correct = [], []
    for f in files:
        z = np.load(f)
        mnli_correct.append(z["mnli_logits"].argmax(1) == meta["mnli_label"])
        pred_ent = z["hans_logits"].argmax(1) == ENT
        hans_correct.append(pred_ent == gold_ent)
    mnli_correct = np.array(mnli_correct); hans_correct = np.array(hans_correct)

    # per-model table
    table = pd.DataFrame({"model": ids, "mnli_acc": mnli_correct.mean(1),
                          "hans_acc": hans_correct.mean(1)})
    for h in ["lexical_overlap", "subsequence", "constituent"]:
        for g, gname in [(True, "ent"), (False, "nonent")]:
            m = (heur == h) & (gold_ent == g)
            table[f"{h}_{gname}"] = hans_correct[:, m].mean(1)
    for s in sorted(set(sub)):
        table[f"sub_{s}"] = hans_correct[:, sub == s].mean(1)
    table.to_csv(out / "per_model_accuracy.csv", index=False)

    # ---- primary outcome: lexical-overlap non-entailment accuracy
    lo_mask = (heur == "lexical_overlap") & ~gold_ent
    Y = hans_correct[:, lo_mask]            # models x 5000
    lo_sub = sub[lo_mask]
    y = Y.mean(1)
    n_items = Y.shape[1]
    rng = np.random.default_rng(args.seed)

    # (1) spread vs. binomial sampling noise
    p = y.mean()
    samp_sd = np.sqrt(p * (1 - p) / n_items)
    between_sd = y.std(ddof=1)
    rng_ = y.max() - y.min()

    # (2) stratified random split-half reliability (each subcase split in half)
    rs = []
    sub_idx = {s: np.where(lo_sub == s)[0] for s in np.unique(lo_sub)}
    for _ in range(args.n_splits):
        a = np.zeros(n_items, bool)
        for s, ix in sub_idx.items():
            perm = rng.permutation(ix)
            a[perm[: len(ix) // 2]] = True
        rs.append(pearsonr(Y[:, a].mean(1), Y[:, ~a].mean(1))[0])
    rs = np.array(rs)
    r_half = float(np.mean(rs))
    r_full = 2 * r_half / (1 + r_half)    # Spearman-Brown: reliability of the full 5000-item score

    # (3) stricter: subcase split (do constructions agree about which models generalise?)
    subs = sorted(sub_idx)
    sub_acc = np.stack([Y[:, sub_idx[s]].mean(1) for s in subs], 1)
    sub_corr = pd.DataFrame(np.corrcoef(sub_acc.T), index=subs, columns=subs)
    part_r = []
    for k in (2,):
        for A in itertools.combinations(range(len(subs)), k):
            B = [j for j in range(len(subs)) if j not in A]
            part_r.append(pearsonr(sub_acc[:, list(A)].mean(1), sub_acc[:, B].mean(1))[0])

    # (4) diagnostics for later steps
    mnli_acc = mnli_correct.mean(1)
    r_mnli = pearsonr(mnli_acc, y)[0]
    rho_mnli = spearmanr(mnli_acc, y)[0]
    bc = bimodality_coefficient(y)

    g_rel = r_half >= GATE_SPLIT_HALF_R
    g_spread = (between_sd >= GATE_SD_RATIO * samp_sd) and (rng_ >= GATE_RANGE)
    lo_, hi_ = MCCOY_MNLI_RANGE

    lines = [
        f"STEP 3 ANALYSIS  ({len(ids)} models)",
        "",
        f"MNLI matched-dev accuracy: min {mnli_acc.min():.4f}  max {mnli_acc.max():.4f}  "
        f"(McCoy et al. report {lo_:.3f}-{hi_:.3f})",
        "",
        f"Primary outcome: HANS lexical-overlap non-entailment accuracy (n={n_items} items/model)",
        f"  mean {y.mean():.4f}  SD {between_sd:.4f}  min {y.min():.4f}  max {y.max():.4f}  "
        f"IQR {np.percentile(y,25):.4f}-{np.percentile(y,75):.4f}",
        f"  binomial sampling SD at this mean: {samp_sd:.4f}  ->  between-model/sampling SD ratio "
        f"{between_sd/samp_sd:.1f}",
        "",
        f"Reliability (stratified random split-half, {args.n_splits} splits):",
        f"  raw split-half r  mean {r_half:.3f}  [2.5%-97.5%: {np.percentile(rs,2.5):.3f}-"
        f"{np.percentile(rs,97.5):.3f}]",
        f"  Spearman-Brown full-length reliability {r_full:.3f}",
        f"  -> ceiling for any predictor: R^2 <= {r_full:.3f}, |r| <= {np.sqrt(r_full):.3f}",
        "",
        "Stricter: agreement across constructions (2-vs-3 subcase partitions, r):",
        "  " + "  ".join(f"{v:.2f}" for v in part_r) + f"   (mean {np.mean(part_r):.3f})",
        "  Between-subcase correlation matrix:",
        sub_corr.round(2).to_string(),
        "",
        "Diagnostics for step 4:",
        f"  MNLI accuracy vs outcome: Pearson {r_mnli:.3f}, Spearman {rho_mnli:.3f}  "
        f"(R^2 = {r_mnli**2:.3f}; share of reliable variance = {r_mnli**2 / r_full:.2f})",
        f"  bimodality coefficient {bc:.3f}  (>0.555 suggests two clusters, cf. Juneja's two basins)",
        "",
        "GATES (pre-registered):",
        f"  G1 reliability: raw split-half r >= {GATE_SPLIT_HALF_R}  ->  {'PASS' if g_rel else 'FAIL'}",
        f"  G2 spread: SD ratio >= {GATE_SD_RATIO} and range >= {GATE_RANGE}  ->  "
        f"{'PASS' if g_spread else 'FAIL'}",
        "",
        ("VERDICT: outcome is real; proceed to step 4 (baselines)." if (g_rel and g_spread) else
         "VERDICT: outcome fails a gate; OOD differences are mostly noise or too small. Stop or "
         "change the outcome before building predictors."),
    ]
    if r_mnli ** 2 / r_full > 0.5:
        lines.append("NOTE: MNLI accuracy alone already explains >50% of the reliable variance; "
                     "step 4 headroom may be small.")
    report = "\n".join(lines)
    print(report)
    (out / "step3_report.txt").write_text(report + "\n")
    with open(out / "step3_summary.json", "w") as f:
        json.dump(dict(n_models=len(ids), outcome_mean=float(y.mean()), outcome_sd=float(between_sd),
                       outcome_min=float(y.min()), outcome_max=float(y.max()),
                       sampling_sd=float(samp_sd), split_half_r=r_half, spearman_brown=r_full,
                       subcase_partition_r=[float(v) for v in part_r], mnli_pearson=float(r_mnli),
                       bimodality_coef=float(bc), gate_reliability=bool(g_rel),
                       gate_spread=bool(g_spread)), f, indent=2)


# ----------------------------------------------------------------------------- misc

def env_info():
    info = {"python": platform.python_version(), "platform": platform.platform()}
    for mod in ["torch", "transformers", "numpy", "scipy", "huggingface_hub", "datasets"]:
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            info[mod] = None
    try:
        import torch
        info["cuda"] = torch.version.cuda
        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    except Exception:
        pass
    return info


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sp = ap.add_subparsers(dest="cmd", required=True)
    for name in ["check", "eval", "analyze"]:
        p = sp.add_parser(name)
        p.add_argument("--out_dir", default="feather_out")
        p.add_argument("--seed", type=int, default=0)
        if name in ("check", "eval"):
            p.add_argument("--models", nargs="+", default=["0", "1", "2", "3", "4"] if name == "check" else ["all"])
            p.add_argument("--device", default="auto")
            p.add_argument("--batch_size", type=int, default=256)
            p.add_argument("--mnli_tsv", default=None, help="GLUE dev_matched.tsv (else HF datasets)")
            p.add_argument("--local_tmpl", default=None,
                           help="Local model dirs, e.g. /models/feather_berts_{i} (skips the Hub)")
        if name == "check":
            p.add_argument("--n_reproduce", type=int, default=2000,
                           help="stored examples to re-run for the logit comparison")
        if name == "eval":
            p.add_argument("--hans_path", default="data/heuristics_evaluation_set.txt")
            p.add_argument("--purge", action="store_true",
                           help="delete each model from the HF cache after use (saves ~44 GB)")
        if name == "analyze":
            p.add_argument("--n_splits", type=int, default=1000)
    args = ap.parse_args()
    {"check": cmd_check, "eval": cmd_eval, "analyze": cmd_analyze}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile feather_step4.py
#!/usr/bin/env python3
"""
Step 4: do internal (mechanistic) signals predict HANS lexical-overlap generalisation better than
in-distribution behaviour, weight-space and representational baselines?

Needs feather_eval.py (steps 2-3) in the same folder and its output folder (feather_out/ with
meta.npz and logits/).

Subcommands
-----------
  split    Freeze a 70/30 split of the 100 models (seed fixed). Do this ONCE, before anything else.
           The 30 test models are not looked at until `analyze --final`.
  extract  GPU. For every model, compute features from MNLI matched-dev inputs ONLY (never HANS):
             - representations (mean-pooled hidden states at 4 depths) on a 512-item probe set
             - attention statistics: attention from hypothesis tokens to identical premise tokens
             - head-ablation profile: mean-ablate each head, change in entailment log-odds on
               high-overlap and on ordinary probe items
             - weights: per-block distance from pretrained BERT, a 4096-d CountSketch of the
               fine-tuning delta, and the classifier weights
           Resumable; --purge deletes each model from the HF cache afterwards.
  analyze  CPU. Kernel ridge regression from each feature block to the outcome (several blocks: stacked
           with non-negative weights fitted on leave-one-out predictions). Development
           models: repeated 10-fold cross-validation. With --final: fit on all 70 development
           models and score once on the 30 test models.

Outcome: HANS lexical-overlap non-entailment accuracy (as in step 3). Secondary: same without the
ln_passive subcase.

Pre-registered decision (primary contrast, on the 30 test models):
  dR2 = R2(baselines + internal) - R2(baselines) >= 0.10 and its 95% bootstrap CI excludes 0
  -> GO.  Internal signals predict but dR2 < 0.10 or CI includes 0 -> PIVOT.  Nothing beats the
  MNLI-accuracy-only model -> STOP.
"""

import argparse
import json
import re
import sys
import time
import zlib
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import feather_eval as fe  # noqa: E402

N_PROBE_HO = 256        # high-overlap probe items
N_PROBE_RAND = 256      # ordinary probe items
HO_THRESH = 0.8         # share of hypothesis words that also occur in the premise
SKETCH_D = 4096
SPLIT_SEED = 2026
N_TEST = 30
PRETRAINED = "google-bert/bert-base-uncased"
MIN_DELTA_R2 = 0.10     # pre-registered GO threshold


# ----------------------------------------------------------------------------- helpers

def word_overlap(p, h):
    hw = re.findall(r"\w+", h.lower())
    pw = set(re.findall(r"\w+", p.lower()))
    return float(np.mean([w in pw for w in hw])) if hw else 0.0


def ent_logodds(logits):
    """log P(entailment) - log P(non-entailment), from 3-way logits (numpy or torch)."""
    import torch
    t = torch.as_tensor(logits)
    others = [k for k in range(3) if k != fe.ENT]
    return (t[..., fe.ENT] - torch.logsumexp(t[..., others], -1))


# ----------------------------------------------------------------------------- split

def cmd_split(args):
    out = Path(args.out_dir)
    f = out / "split.json"
    if f.exists() and not args.force:
        print(f"{f} already exists; refusing to overwrite (that would leak the test set).")
        print(json.loads(f.read_text()))
        return
    perm = np.random.default_rng(SPLIT_SEED).permutation(fe.N_MODELS)
    split = {"seed": SPLIT_SEED, "test": sorted(int(i) for i in perm[:N_TEST]),
             "dev": sorted(int(i) for i in perm[N_TEST:]),
             "created": time.strftime("%Y-%m-%d %H:%M:%S")}
    out.mkdir(parents=True, exist_ok=True)
    f.write_text(json.dumps(split, indent=1))
    print(f"Wrote {f}: {len(split['dev'])} dev, {len(split['test'])} test models.")


# ----------------------------------------------------------------------------- probe set

def build_probe(mnli, out):
    """Fixed probe set from MNLI matched dev: high-overlap items + ordinary items."""
    fdir = out / "features"; fdir.mkdir(parents=True, exist_ok=True)
    pf = fdir / "probe.json"
    ov = np.array([word_overlap(p, h) for p, h in zip(mnli["premise"], mnli["hypothesis"])])
    np.save(fdir / "mnli_overlap.npy", ov)
    if pf.exists():
        probe = json.loads(pf.read_text())
    else:
        rng = np.random.default_rng(0)
        ho = np.where(ov >= HO_THRESH)[0]; rest = np.where(ov < HO_THRESH)[0]
        ho_sel = np.sort(rng.choice(ho, min(N_PROBE_HO, len(ho)), replace=False))
        rd_sel = np.sort(rng.choice(rest, min(N_PROBE_RAND, len(rest)), replace=False))
        probe = {"idx": [int(i) for i in np.concatenate([ho_sel, rd_sel])],
                 "is_ho": [True] * len(ho_sel) + [False] * len(rd_sel),
                 "n_ho_in_dev": int(len(ho)),
                 "n_ho_nonent_in_dev": int(((ov >= HO_THRESH) & (mnli["label"] != fe.ENT)).sum())}
        pf.write_text(json.dumps(probe))
    print(f"MNLI dev: {probe['n_ho_in_dev']} high-overlap items "
          f"({probe['n_ho_nonent_in_dev']} non-entailment); probe set = {len(probe['idx'])} items")
    return probe


# ----------------------------------------------------------------------------- extract

def get_layers(model):
    return model.bert.encoder.layer


def encode_batches(tok, prem, hyp, device, bs):
    order = np.argsort([len(p) + len(h) for p, h in zip(prem, hyp)])
    batches = []
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok([prem[j] for j in idx], [hyp[j] for j in idx], truncation=True,
                  max_length=fe.MAX_LEN, padding=True, return_tensors="pt")
        batches.append((idx, {k: v.to(device) for k, v in enc.items()}))
    return batches


def extract_one(i, args, device, probe_text, is_ho, pre_sd):
    import torch
    from transformers import AutoTokenizer, BertForSequenceClassification
    path, sha = fe.model_source(i, args)
    tok = AutoTokenizer.from_pretrained(path)
    try:
        model = BertForSequenceClassification.from_pretrained(path, attn_implementation="eager")
    except (TypeError, ValueError):
        model = BertForSequenceClassification.from_pretrained(path)
    model = model.to(device).eval()
    cfg = model.config
    L, NH, H = cfg.num_hidden_layers, cfg.num_attention_heads, cfg.hidden_size
    dh = H // NH
    layers = get_layers(model)
    rep_layers = [L // 4, L // 2, (3 * L) // 4, L]
    N = len(probe_text[0])
    batches = encode_batches(tok, probe_text[0], probe_text[1], device, args.batch_size)
    is_ho_t = torch.tensor(is_ho, device=device)
    cls_id, sep_id = tok.cls_token_id, tok.sep_token_id

    # ---- clean pass: logits, representations, attention stats, per-layer context means
    ctx_sum = torch.zeros(L, H, device=device, dtype=torch.float64); ntok = 0
    hooks, cur_mask = [], {}

    def make_capture(l):
        def hook(mod, inp):
            x = inp[0]; m = cur_mask["m"].unsqueeze(-1).to(x.dtype)
            ctx_sum[l] += (x * m).sum((0, 1)).double()
        return hook

    for l in range(L):
        hooks.append(layers[l].attention.output.dense.register_forward_pre_hook(make_capture(l)))
    clean_lo = torch.empty(N, device=device)
    rep = torch.empty(len(rep_layers), N, H, device=device)
    match_sum = torch.zeros(L, NH, device=device, dtype=torch.float64); match_n = 0
    prem_sum = torch.zeros(L, NH, device=device, dtype=torch.float64); prem_n = 0
    with torch.inference_mode():
        for idx, enc in batches:
            am = enc["attention_mask"]; cur_mask["m"] = am; ntok += int(am.sum())
            o = model(**enc, output_hidden_states=True, output_attentions=True)
            it = torch.as_tensor(idx, device=device)
            clean_lo[it] = ent_logodds(o.logits.float())
            amf = am.unsqueeze(-1).float()
            for r, l in enumerate(rep_layers):
                hs = o.hidden_states[l].float()
                rep[r, it] = (hs * amf).sum(1) / amf.sum(1)
            # attention statistics on high-overlap items only
            ids, tt = enc["input_ids"], enc["token_type_ids"]
            special = (ids == cls_id) | (ids == sep_id) | (am == 0)
            if tok.unk_token_id is not None:
                special = special | (ids == tok.unk_token_id)
            hypq = (tt == 1) & ~special
            premk = (tt == 0) & ~special
            match = (ids.unsqueeze(2) == ids.unsqueeze(1)) & hypq.unsqueeze(2) & premk.unsqueeze(1)
            sel = is_ho_t[it]
            q_has = match.any(-1) & sel.unsqueeze(1)                 # (B,T) queries with a match
            q_hyp = hypq & sel.unsqueeze(1)
            att = torch.stack([a.float() for a in o.attentions], 1)  # (B,L,NH,T,T)
            mass_match = (att * match[:, None, None].float()).sum(-1)  # (B,L,NH,T)
            mass_prem = (att * premk[:, None, None, None, :].float()).sum(-1)
            match_sum += (mass_match * q_has[:, None, None].float()).sum((0, 3)).double()
            match_n += int(q_has.sum())
            prem_sum += (mass_prem * q_hyp[:, None, None].float()).sum((0, 3)).double()
            prem_n += int(q_hyp.sum())
            del o, att
    for hk in hooks:
        hk.remove()
    ctx_mean = (ctx_sum / ntok).float()
    attn_match = (match_sum / max(match_n, 1)).float().cpu().numpy()
    attn_prem = (prem_sum / max(prem_n, 1)).float().cpu().numpy()

    # ---- head-ablation profile (mean ablation of each head's context vector)
    abl_ho = np.zeros((L, NH), np.float32); abl_rd = np.zeros((L, NH), np.float32)
    ho_np = np.array(is_ho)
    base = clean_lo.cpu().numpy()
    use_amp = device == "cuda" and not args.fp32_ablation

    def run_lo():
        lo = np.empty(N, np.float32)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            for idx, enc in batches:
                lo[idx] = ent_logodds(model(**enc).logits.float()).cpu().numpy()
        return lo

    with torch.inference_mode():
        base_abl = run_lo()  # clean reference at the same precision as the ablated runs
        amp_err = float(np.abs(base_abl - base).max())
        for l in range(L):
            for h in range(NH):
                sl = slice(h * dh, (h + 1) * dh)

                def hook(mod, inp, sl=sl, l=l):
                    x = inp[0].clone(); x[..., sl] = ctx_mean[l, sl].to(x.dtype)
                    return (x,)
                hk = layers[l].attention.output.dense.register_forward_pre_hook(hook)
                lo = run_lo()
                hk.remove()
                d = lo - base_abl
                abl_ho[l, h] = d[ho_np].mean() if ho_np.any() else 0.0
                abl_rd[l, h] = d[~ho_np].mean()

    # ---- weights: distance from pretrained, CountSketch of the delta, classifier
    ft = model.bert.state_dict()
    groups = ["embeddings"] + [f"layer.{k}" for k in range(L)] + ["pooler"]
    gsq = dict.fromkeys(groups, 0.0); gsq0 = dict.fromkeys(groups, 0.0)
    sketch = torch.zeros(SKETCH_D, device=device, dtype=torch.float64)
    with torch.inference_mode():
        for key in sorted(ft):
            w = ft[key]
            if key not in pre_sd or not torch.is_floating_point(w) or w.shape != pre_sd[key].shape:
                continue
            w0 = pre_sd[key].to(device=device, dtype=torch.float32)
            d = (w.float() - w0).flatten()
            g = next((gr for gr in groups if f"{gr}." in key or key.startswith(gr)), None)
            if g is not None:
                gsq[g] += float((d.double() ** 2).sum()); gsq0[g] += float((w0.double() ** 2).sum())
            gen = torch.Generator(device=device).manual_seed(zlib.crc32(key.encode()))
            b = torch.randint(0, SKETCH_D, (d.numel(),), generator=gen, device=device)
            s = torch.randint(0, 2, (d.numel(),), generator=gen, device=device).float() * 2 - 1
            sketch.index_add_(0, b, (d * s).double())
    dnorm = np.array([np.sqrt(gsq[g]) for g in groups], np.float32)
    drel = np.array([np.sqrt(gsq[g] / gsq0[g]) if gsq0[g] > 0 else 0 for g in groups], np.float32)
    clf = torch.cat([model.classifier.weight.flatten(), model.classifier.bias]).detach().float().cpu().numpy()

    res = dict(rep=rep.cpu().numpy().astype(np.float16), rep_layers=np.array(rep_layers),
               attn_match=attn_match, attn_prem=attn_prem, abl_ho=abl_ho, abl_rand=abl_rd,
               clean_lo=base, delta_norm=dnorm, delta_rel=drel, groups=np.array(groups),
               sketch=sketch.float().cpu().numpy(), clf=clf, sha=sha, device=device,
               amp_max_err=amp_err)
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return res


def cmd_extract(args):
    import torch
    from transformers import BertModel
    device = fe.pick_device(args.device)
    out = Path(args.out_dir); fdir = out / "features"; fdir.mkdir(parents=True, exist_ok=True)
    meta = np.load(out / "meta.npz", allow_pickle=True)
    mnli = fe.load_mnli(args.mnli_tsv)
    assert np.array_equal(mnli["label"], meta["mnli_label"]), \
        "MNLI order/labels differ from step 3 - cannot align with saved logits"
    probe = build_probe(mnli, out)
    pidx = probe["idx"]
    probe_text = ([mnli["premise"][j] for j in pidx], [mnli["hypothesis"][j] for j in pidx])
    print(f"Loading pretrained reference {args.pretrained} ...")
    pre_sd = {k: v.detach().cpu() for k, v in BertModel.from_pretrained(args.pretrained).state_dict().items()}
    models = fe.parse_models(args.models)
    print(f"device={device} models={len(models)}")
    failed = {}
    for k, i in enumerate(models):
        f = fdir / f"feat_{i:03d}.npz"
        if f.exists():
            print(f"[{k+1}/{len(models)}] model {i}: exists, skipping"); continue
        t0 = time.time()
        try:
            res = extract_one(i, args, device, probe_text, probe["is_ho"], pre_sd)
        except Exception as e:
            failed[i] = f"{type(e).__name__}: {e}"
            print(f"[{k+1}/{len(models)}] model {i}: FAILED ({failed[i][:300]})")
            if device == "cuda":
                torch.cuda.empty_cache()
            continue
        tmp = f.with_suffix(".tmp.npz"); np.savez(tmp, **res); tmp.replace(f)
        print(f"[{k+1}/{len(models)}] model {i}: ok  max|abl_ho|={np.abs(res['abl_ho']).max():.3f}  "
              f"fp16 err={res['amp_max_err']:.1e}  "
              f"({time.time()-t0:.0f}s)")
        if args.purge and not args.local_tmpl:
            fe.purge_from_cache(i)
    (out / "step4_extract_env.json").write_text(json.dumps({**fe.env_info(), "failed": failed}, indent=2))
    done = len(list(fdir.glob("feat_*.npz")))
    print(f"\nDone: {done} models have features. Failed this run: {sorted(failed) or 'none'}")


# ----------------------------------------------------------------------------- analyze

def center_kernels(Ktr, Kte):
    n = Ktr.shape[0]
    mr = Ktr.mean(0); ma = Ktr.mean()
    Ktr_c = Ktr - mr[None] - mr[:, None] + ma
    Kte_c = Kte - mr[None] - Kte.mean(1, keepdims=True) + ma
    return Ktr_c, Kte_c


ALPHAS = np.logspace(-4, 3, 22)


def krr(Ktr, Kte, ytr):
    """Kernel ridge with intercept; alpha by exact leave-one-out on the training models."""
    Ktr_c, Kte_c = center_kernels(Ktr, Kte)
    yc = ytr - ytr.mean()
    w, V = np.linalg.eigh(Ktr_c)
    w = np.clip(w, 0, None)
    best = (np.inf, None, None)
    Vy = V.T @ yc
    for a in ALPHAS:
        Hdiag = (V ** 2) @ (w / (w + a)) + 1.0 / len(ytr)  # + the intercept's share of the hat matrix
        fit = V @ (w / (w + a) * Vy)
        res = (yc - fit) / np.clip(1 - Hdiag, 1e-6, None)   # exact leave-one-out residuals
        if (res ** 2).mean() < best[0]:
            best = ((res ** 2).mean(), a, ytr - res)
    a = best[1]
    coef = V @ (Vy / (w + a))
    return ytr.mean() + Kte_c @ coef, a, best[2]


def r2(y, p):
    return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()


def linear_kernel_block(X):
    """Returns f(tr, rows) -> kernel using train-fold standardisation."""
    X = np.asarray(X, np.float64)

    def f(tr):
        mu = X[tr].mean(0); sd = X[tr].std(0); keep = sd > 1e-12
        Z = (X[:, keep] - mu[keep]) / sd[keep]
        K = Z @ Z.T / max(keep.sum(), 1)
        return K / np.mean(np.diag(K)[tr])
    return f


def rbf_distance_block(X):
    """Juneja-style weight geometry: Gaussian kernel on Euclidean distances between models'
    weights (the CountSketch of the fine-tuning delta preserves these distances). Juneja et al.
    report Euclidean distance behaves similarly to their convexity-gap metric. Bandwidth =
    median distance among the training models only."""
    X = np.asarray(X, np.float64)
    D2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)

    def f(tr):
        d = D2[np.ix_(tr, tr)][np.triu_indices(len(tr), 1)]
        return np.exp(-D2 / np.median(d))
    return f


def fixed_kernel_block(K):
    def f(tr):
        return K / np.mean(np.diag(K)[tr])
    return f


def linear_cka_kernel(reps):
    """reps: models x N x H -> models x models linear CKA."""
    F = []
    for X in reps:
        X = X.astype(np.float64); X = X - X.mean(0)
        G = X @ X.T
        F.append((G / np.linalg.norm(G)).ravel())
    F = np.array(F)
    return F @ F.T


def predict_set(blocks, names, tr, te, y):
    """One block: kernel ridge. Several blocks: stacking - a kernel ridge per block, then
    non-negative weights fitted to the blocks' leave-one-out predictions on the training models,
    so one strong block is not diluted by weak ones."""
    if len(names) == 1:
        K = blocks[names[0]](tr)
        return krr(K[np.ix_(tr, tr)], K[np.ix_(te, tr)], y[tr])[0]
    from scipy.optimize import nnls
    P_loo, P_te = [], []
    for n in names:
        K = blocks[n](tr)
        pte, _, ploo = krr(K[np.ix_(tr, tr)], K[np.ix_(te, tr)], y[tr])
        P_loo.append(ploo); P_te.append(pte)
    P_loo = np.array(P_loo).T; P_te = np.array(P_te).T
    mu = P_loo.mean(0); ym = y[tr].mean()
    wts, _ = nnls(P_loo - mu, y[tr] - ym)
    return ym + (P_te - mu) @ wts


def cv_oof(blocks, names, idx, y, n_rep, seed):
    """Repeated 10-fold CV on the model indices `idx`; returns mean OOF prediction and R2s."""
    rng = np.random.default_rng(seed)
    preds = np.zeros((n_rep, len(idx)))
    for r in range(n_rep):
        folds = np.array_split(rng.permutation(len(idx)), 10)
        for fo in folds:
            te = idx[fo]; tr = idx[np.setdiff1d(np.arange(len(idx)), fo)]
            preds[r, fo] = predict_set(blocks, names, tr, te, y)
    r2s = np.array([r2(y[idx], p) for p in preds])
    return preds.mean(0), r2s


def boot_delta(y, pa, pb, n=4000, seed=0):
    rng = np.random.default_rng(seed); m = len(y); d = []
    for _ in range(n):
        b = rng.integers(0, m, m)
        if np.var(y[b]) == 0:
            continue
        d.append(r2(y[b], pa[b]) - r2(y[b], pb[b]))
    return np.percentile(d, [2.5, 97.5])


def build_blocks(out, ids):
    meta = np.load(out / "meta.npz", allow_pickle=True)
    fdir = out / "features"
    ov = np.load(fdir / "mnli_overlap.npy")
    lab = meta["mnli_label"]
    gold, heur, sub = meta["hans_gold_ent"], meta["hans_heuristic"], meta["hans_subcase"]
    lo_mask = (heur == "lexical_overlap") & ~gold
    lo_np = lo_mask & (sub != "ln_passive")
    ho = ov >= HO_THRESH
    y, y_np, idsum, iditems, wts, attn, abl, reps, mnli_acc = [], [], [], [], [], [], [], [], []
    sketch = []
    for i in ids:
        z = np.load(out / "logits" / f"model_{i:03d}.npz")
        hl = z["hans_logits"]; ml = z["mnli_logits"]
        y.append((hl[lo_mask].argmax(1) != fe.ENT).mean())
        y_np.append((hl[lo_np].argmax(1) != fe.ENT).mean())
        pr = np.exp(ml - ml.max(1, keepdims=True)); pr /= pr.sum(1, keepdims=True)
        pred = ml.argmax(1); corr = pred == lab
        rec = [corr[lab == c].mean() for c in range(3)]
        ent = -(pr * np.log(pr + 1e-12)).sum(1)
        hn = ho & (lab != fe.ENT); he = ho & (lab == fe.ENT)
        mnli_acc.append(corr.mean())
        idsum.append([corr.mean(), *rec, pr.max(1).mean(), ent.mean(), pr[:, fe.ENT].mean(),
                      corr[ho].mean(), corr[hn].mean(), pr[hn, fe.ENT].mean(), pr[he, fe.ENT].mean()])
        iditems.append(np.log(pr[ho, fe.ENT] + 1e-9) - np.log(1 - pr[ho, fe.ENT] + 1e-9))
        f = np.load(fdir / f"feat_{i:03d}.npz")
        wts.append(np.concatenate([f["delta_norm"], f["delta_rel"], f["sketch"], f["clf"]]))
        sketch.append(f["sketch"])
        attn.append(np.concatenate([f["attn_match"].ravel(), f["attn_prem"].ravel()]))
        abl.append(np.concatenate([f["abl_ho"].ravel(), f["abl_rand"].ravel()]))
        reps.append(f["rep"])
    reps = np.array(reps)  # models x layers x N x H
    cka = np.mean([linear_cka_kernel(reps[:, l]) for l in range(reps.shape[1])], 0)
    blocks = {
        "MNLI-acc": linear_kernel_block(np.array(mnli_acc)[:, None]),
        "ID-summary": linear_kernel_block(idsum),
        "ID-overlap-items": linear_kernel_block(iditems),
        "Weights": linear_kernel_block(wts),
        "Repr-CKA": fixed_kernel_block(cka),
        "Weight-geometry": rbf_distance_block(sketch),
        "Attn-stats": linear_kernel_block(attn),
        "Head-ablation": linear_kernel_block(abl),
    }
    extra = dict(n_ho=int(ho.sum()), n_ho_nonent=int((ho & (lab != fe.ENT)).sum()))
    return blocks, np.array(y), np.array(y_np), extra


BASE = ["ID-summary", "ID-overlap-items", "Weights", "Weight-geometry", "Repr-CKA"]
INTERNAL = ["Attn-stats", "Head-ablation"]
SETS = {
    "MNLI-acc only": ["MNLI-acc"],
    "ID-summary": ["ID-summary"],
    "ID-overlap-items": ["ID-overlap-items"],
    "Weights": ["Weights"],
    "Weight-geometry (Juneja-style)": ["Weight-geometry"],
    "Repr-CKA": ["Repr-CKA"],
    "Attn-stats": ["Attn-stats"],
    "Head-ablation": ["Head-ablation"],
    "INTERNAL (attn+ablation)": INTERNAL,
    "ALL BASELINES": BASE,
    "ALL BASELINES + INTERNAL": BASE + INTERNAL,
}


def cmd_analyze(args):
    out = Path(args.out_dir)
    split = json.loads((out / "split.json").read_text())
    have = {int(p.stem.split("_")[1]) for p in (out / "features").glob("feat_*.npz")}
    dev = [i for i in split["dev"] if i in have]; test = [i for i in split["test"] if i in have]
    miss = sorted(set(split["dev"] + split["test"]) - have)
    ids = dev + test
    blocks, y_all, ynp_all, extra = build_blocks(out, ids)
    dev_idx = np.arange(len(dev)); test_idx = np.arange(len(dev), len(ids))
    lines = [f"STEP 4 ANALYSIS  ({'FINAL: test models' if args.final else 'development models, CV'})",
             f"models: {len(dev)} dev, {len(test)} test" + (f"; MISSING features: {miss}" if miss else ""),
             f"MNLI high-overlap items (overlap >= {HO_THRESH}): {extra['n_ho']} "
             f"({extra['n_ho_nonent']} non-entailment)", ""]
    summary = {}
    for oname, y in [("HANS-LO non-entailment (primary)", y_all),
                     ("HANS-LO non-entailment without ln_passive (secondary)", ynp_all)]:
        lines.append(f"== Outcome: {oname}")
        preds = {}
        if not args.final:
            lines.append(f"{'feature set':32s} {'CV R2 mean':>10s} {'[min, max over repeats]':>24s}")
            for sname, names in SETS.items():
                p, r2s = cv_oof(blocks, names, dev_idx, y, args.n_rep, args.seed)
                preds[sname] = p
                lines.append(f"{sname:32s} {r2s.mean():10.3f}   [{r2s.min():.3f}, {r2s.max():.3f}]")
            yy = y[dev_idx]
        else:
            lines.append(f"{'feature set':32s} {'test R2':>10s}")
            for sname, names in SETS.items():
                preds[sname] = predict_set(blocks, names, dev_idx, test_idx, y)
                lines.append(f"{sname:32s} {r2(y[test_idx], preds[sname]):10.3f}")
            yy = y[test_idx]
        A, B = "ALL BASELINES + INTERNAL", "ALL BASELINES"
        d = r2(yy, preds[A]) - r2(yy, preds[B]); ci = boot_delta(yy, preds[A], preds[B])
        d2 = r2(yy, preds["INTERNAL (attn+ablation)"]) - r2(yy, preds["ID-overlap-items"])
        ci2 = boot_delta(yy, preds["INTERNAL (attn+ablation)"], preds["ID-overlap-items"])
        lines += ["",
                  f"PRIMARY CONTRAST  dR2 = R2(baselines+internal) - R2(baselines) = {d:+.3f}  "
                  f"95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]",
                  f"secondary         R2(internal) - R2(ID-overlap-items)          = {d2:+.3f}  "
                  f"95% CI [{ci2[0]:+.3f}, {ci2[1]:+.3f}]"]
        r2_int = r2(yy, preds["INTERNAL (attn+ablation)"])
        r2_acc = r2(yy, preds["MNLI-acc only"])
        if d >= MIN_DELTA_R2 and ci[0] > 0:
            verdict = "GO - internal signals add predictive information beyond all baselines."
        elif r2_int >= max(0.05, r2_acc + 0.05):
            verdict = ("PIVOT - internal signals predict the outcome but do not clear the "
                       f"pre-registered bar over baselines (need dR2 >= {MIN_DELTA_R2} with CI > 0).")
        else:
            verdict = ("STOP - internal signals predict no better than MNLI accuracy alone "
                       "(check whether a baseline does: that would be a different paper).")
        lines += [f"VERDICT ({'binding' if args.final else 'provisional, dev CV only'}): {verdict}", ""]
        summary[oname] = dict(delta_r2=float(d), ci=[float(c) for c in ci],
                              r2={k: float(r2(yy, p)) for k, p in preds.items()}, verdict=verdict)
    if args.final:
        lk = out / "step4_test_unlocked.json"
        prev = json.loads(lk.read_text()) if lk.exists() else []
        if prev:
            lines.insert(1, f"WARNING: test set already scored {len(prev)} time(s) before; "
                            "only the first look is confirmatory.")
        prev.append(time.strftime("%Y-%m-%d %H:%M:%S")); lk.write_text(json.dumps(prev))
    rep = "\n".join(lines); print(rep)
    tag = "final" if args.final else "dev"
    (out / f"step4_report_{tag}.txt").write_text(rep + "\n")
    (out / f"step4_summary_{tag}.json").write_text(json.dumps(summary, indent=2))


# ----------------------------------------------------------------------------- main

def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sp = ap.add_subparsers(dest="cmd", required=True)
    for name in ["split", "extract", "analyze"]:
        p = sp.add_parser(name)
        p.add_argument("--out_dir", default="feather_out")
        p.add_argument("--seed", type=int, default=0)
        if name == "split":
            p.add_argument("--force", action="store_true")
        if name == "extract":
            p.add_argument("--models", nargs="+", default=["all"])
            p.add_argument("--device", default="auto")
            p.add_argument("--batch_size", type=int, default=64)
            p.add_argument("--mnli_tsv", default=None)
            p.add_argument("--local_tmpl", default=None)
            p.add_argument("--pretrained", default=PRETRAINED)
            p.add_argument("--purge", action="store_true")
            p.add_argument("--fp32_ablation", action="store_true",
                           help="run the 144 ablation passes in fp32 (slower; default fp16 on GPU)")
        if name == "analyze":
            p.add_argument("--final", action="store_true",
                           help="score on the 30 held-out test models (do this once)")
            p.add_argument("--n_rep", type=int, default=20)
    args = ap.parse_args()
    {"split": cmd_split, "extract": cmd_extract, "analyze": cmd_analyze}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile feather_causal.py
#!/usr/bin/env python3
"""
Step 5 (exploratory, decided after the binding step-4 PIVOT): are the heads whose ablation
profile PREDICTS HANS generalisation also CAUSALLY involved in it?

Head sets were chosen on the 70 development models only (feather_out/exploratory.json) and are
fixed below. The test is run on the 30 held-out test models, which played no part in choosing them.

Conditions per model (mean ablation of each head's context vector; means from the 512-item MNLI
probe set used in step 4):
  clean
  pred3, pred6   top-3 / top-6 heads by |corr(ablation effect, HANS-LO accuracy)| on dev models
  match3         top-3 heads by |corr(attention to repeated words, HANS-LO accuracy)|
  rand6_0..4     five fixed random sets of 6 heads, excluding the predictive heads

Measured: HANS lexical-overlap non-entailment accuracy (the outcome), HANS lexical-overlap
ENTAILMENT accuracy (to catch ablations that merely suppress "entailment" everywhere), and MNLI
accuracy on 2,000 fixed matched-dev items.

Hypotheses (stated before running):
  H1  |change in HANS-LO non-ent accuracy| is larger for pred6 than for the mean of the rand6 sets
      (paired Wilcoxon over the 30 test models).
  H2  pred6 changes HANS-LO non-ent accuracy more than it changes MNLI accuracy, and more than
      it changes LO-entailment accuracy in the opposite direction (i.e. it is not just a
      global shift away from "entailment").
  H3  (descriptive) the size of the change correlates with the model's clean HANS-LO accuracy.

Usage:  python feather_causal.py run      (GPU; resumable)
        python feather_causal.py analyze  (CPU)
"""

import argparse
import hashlib
import json
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import feather_eval as fe     # noqa: E402
import feather_step4 as s4    # noqa: E402

TEST_MODELS = [2, 3, 10, 12, 14, 15, 16, 17, 19, 25, 29, 31, 35, 44, 46, 47, 48, 51, 55, 64,
               70, 73, 74, 75, 76, 84, 85, 88, 91, 97]
PRED_HEADS = [(7, 8), (8, 4), (5, 5), (9, 8), (4, 4), (7, 0)]   # ranked, from dev models only
MATCH_HEADS = [(8, 1), (8, 4), (10, 1)]
PROBE_MD5 = "1ec2d80f86436efc94f21576c911c351"
N_MNLI = 2000


def head_sets(L=12, NH=12):
    rng = np.random.default_rng(12345)
    excl = set(PRED_HEADS) | set(MATCH_HEADS)
    pool = [(l, h) for l in range(L) for h in range(NH) if (l, h) not in excl]
    sets = {"clean": [], "pred3": PRED_HEADS[:3], "pred6": PRED_HEADS[:6], "match3": MATCH_HEADS}
    for r in range(5):
        pick = rng.choice(len(pool), 6, replace=False)
        sets[f"rand6_{r}"] = [pool[j] for j in sorted(pick)]
    return sets


def cmd_run(args):
    import torch
    from transformers import AutoTokenizer, BertForSequenceClassification
    device = fe.pick_device(args.device)
    out = Path(args.out_dir); rdir = out / "causal"; rdir.mkdir(parents=True, exist_ok=True)
    mnli = fe.load_mnli(args.mnli_tsv)
    probe = s4.build_probe(mnli, out)
    md5 = hashlib.md5(json.dumps(probe["idx"]).encode()).hexdigest()
    if not args.local_tmpl and md5 != PROBE_MD5:
        sys.exit(f"Probe set differs from step 4 ({md5} != {PROBE_MD5}); MNLI data changed?")
    hans = fe.load_hans(args.hans_path)
    lo = hans["heuristic"] == "lexical_overlap"
    lo_n = np.where(lo & ~hans["gold_ent"])[0]; lo_e = np.where(lo & hans["gold_ent"])[0]
    rng = np.random.default_rng(1)
    mi = np.sort(rng.choice(len(mnli["label"]), min(N_MNLI, len(mnli["label"])), replace=False))
    sets = head_sets(args.n_layers, args.n_heads)
    (rdir / "head_sets.json").write_text(json.dumps(sets))
    pidx = probe["idx"]
    models = TEST_MODELS if args.models == ["test"] else fe.parse_models(args.models)
    for k, i in enumerate(models):
        f = rdir / f"model_{i:03d}.json"
        if f.exists():
            print(f"[{k+1}/{len(models)}] model {i}: exists, skipping"); continue
        t0 = time.time()
        try:
            path, sha = fe.model_source(i, args)
            tok = AutoTokenizer.from_pretrained(path)
            model = BertForSequenceClassification.from_pretrained(path).to(device).eval()
            layers = model.bert.encoder.layer
            L, NH, H = (model.config.num_hidden_layers, model.config.num_attention_heads,
                        model.config.hidden_size)
            dh = H // NH
            # per-layer mean context vector over the probe set (same as step 4)
            pb = s4.encode_batches(tok, [mnli["premise"][j] for j in pidx],
                                   [mnli["hypothesis"][j] for j in pidx], device, args.batch_size)
            csum = torch.zeros(L, H, device=device, dtype=torch.float64); nt = 0; cur = {}

            def cap(l):
                def hk(mod, inp):
                    m = cur["m"].unsqueeze(-1).to(inp[0].dtype)
                    csum[l] += (inp[0] * m).sum((0, 1)).double()
                return hk
            hs = [layers[l].attention.output.dense.register_forward_pre_hook(cap(l)) for l in range(L)]
            with torch.inference_mode():
                for _, enc in pb:
                    cur["m"] = enc["attention_mask"]; nt += int(enc["attention_mask"].sum()); model(**enc)
            for h_ in hs:
                h_.remove()
            cmean = (csum / nt).float()

            evals = {
                "lo_nonent": s4.encode_batches(tok, [hans["premise"][j] for j in lo_n],
                                               [hans["hypothesis"][j] for j in lo_n], device, args.batch_size),
                "lo_ent": s4.encode_batches(tok, [hans["premise"][j] for j in lo_e],
                                            [hans["hypothesis"][j] for j in lo_e], device, args.batch_size),
                "mnli": s4.encode_batches(tok, [mnli["premise"][j] for j in mi],
                                          [mnli["hypothesis"][j] for j in mi], device, args.batch_size),
            }
            mlab = mnli["label"][mi]
            res = {"model": i, "sha": sha}
            for cname, heads in sets.items():
                by_layer = {}
                for (l, h) in heads:
                    by_layer.setdefault(l, []).append(h)
                hooks = []
                for l, hh in by_layer.items():
                    def hk(mod, inp, l=l, hh=hh):
                        x = inp[0].clone()
                        for h in hh:
                            x[..., h * dh:(h + 1) * dh] = cmean[l, h * dh:(h + 1) * dh].to(x.dtype)
                        return (x,)
                    hooks.append(layers[l].attention.output.dense.register_forward_pre_hook(hk))
                r = {}
                with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16,
                                                             enabled=(device == "cuda" and not args.fp32)):
                    for ename, batches in evals.items():
                        n = sum(len(ix) for ix, _ in batches); pred = np.empty(n, np.int64)
                        for ix, enc in batches:
                            pred[ix] = model(**enc).logits.float().argmax(-1).cpu().numpy()
                        if ename == "lo_nonent":
                            r[ename] = float((pred != fe.ENT).mean())
                        elif ename == "lo_ent":
                            r[ename] = float((pred == fe.ENT).mean())
                        else:
                            r[ename] = float((pred == mlab).mean())
                for h_ in hooks:
                    h_.remove()
                res[cname] = r
            f.write_text(json.dumps(res))
            c = res["clean"]; p = res["pred6"]
            print(f"[{k+1}/{len(models)}] model {i}: clean LO-nonent {c['lo_nonent']:.3f} -> pred6 "
                  f"{p['lo_nonent']:.3f}; LO-ent {c['lo_ent']:.3f} -> {p['lo_ent']:.3f}; "
                  f"MNLI {c['mnli']:.3f} -> {p['mnli']:.3f}  ({time.time()-t0:.0f}s)")
            del model
            if device == "cuda":
                torch.cuda.empty_cache()
            if args.purge and not args.local_tmpl:
                fe.purge_from_cache(i)
        except Exception as e:
            print(f"[{k+1}/{len(models)}] model {i}: FAILED {type(e).__name__}: {str(e)[:300]}")


def cmd_analyze(args):
    from scipy.stats import spearmanr, wilcoxon
    rdir = Path(args.out_dir) / "causal"
    rows = [json.loads(p.read_text()) for p in sorted(rdir.glob("model_*.json"))]
    if len(rows) < 5:
        sys.exit(f"only {len(rows)} models done")
    conds = [c for c in rows[0] if c not in ("model", "sha")]
    clean = {m: np.array([r["clean"][m] for r in rows]) for m in ("lo_nonent", "lo_ent", "mnli")}
    d = {c: {m: np.array([r[c][m] for r in rows]) - clean[m] for m in clean} for c in conds}
    rand = [c for c in conds if c.startswith("rand6")]
    rand_abs = np.mean([np.abs(d[c]["lo_nonent"]) for c in rand], 0)
    L = [f"CAUSAL CHECK ({len(rows)} held-out test models; head sets chosen on dev models)", "",
         f"{'condition':10s} {'dLO-nonent':>11s} {'|d|':>7s} {'dLO-ent':>9s} {'dMNLI':>8s}   (means over models)"]
    for c in conds:
        L.append(f"{c:10s} {d[c]['lo_nonent'].mean():+11.3f} {np.abs(d[c]['lo_nonent']).mean():7.3f} "
                 f"{d[c]['lo_ent'].mean():+9.3f} {d[c]['mnli'].mean():+8.3f}")
    p6 = np.abs(d["pred6"]["lo_nonent"])
    w = wilcoxon(p6, rand_abs)
    L += ["", f"H1  |dLO-nonent| pred6 {p6.mean():.3f} vs random-6 mean {rand_abs.mean():.3f}: "
              f"Wilcoxon p = {w.pvalue:.2g}  -> {'SUPPORTED' if (w.pvalue < 0.05 and p6.mean() > rand_abs.mean()) else 'not supported'}"]
    ratio_mnli = np.abs(d["pred6"]["lo_nonent"]).mean() / max(np.abs(d["pred6"]["mnli"]).mean(), 1e-6)
    net = d["pred6"]["lo_nonent"] + d["pred6"]["lo_ent"]   # >0: better at both; ~0: pure label shift
    L += [f"H2  |dLO-nonent| / |dMNLI| for pred6 = {ratio_mnli:.1f};  dLO-nonent + dLO-ent = "
          f"{net.mean():+.3f} (≈0 would mean a pure shift away from 'entailment')"]
    rho = spearmanr(clean["lo_nonent"], d["pred6"]["lo_nonent"])
    L += [f"H3  Spearman(clean LO-nonent, dLO-nonent under pred6) = {rho.correlation:+.2f} (p = {rho.pvalue:.2g})"]
    rep = "\n".join(L); print(rep)
    (Path(args.out_dir) / "causal_report.txt").write_text(rep + "\n")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sp = ap.add_subparsers(dest="cmd", required=True)
    for name in ["run", "analyze"]:
        p = sp.add_parser(name)
        p.add_argument("--out_dir", default="feather_out")
        if name == "run":
            p.add_argument("--models", nargs="+", default=["test"])
            p.add_argument("--device", default="auto")
            p.add_argument("--batch_size", type=int, default=128)
            p.add_argument("--mnli_tsv", default=None)
            p.add_argument("--hans_path", default="data/heuristics_evaluation_set.txt")
            p.add_argument("--local_tmpl", default=None)
            p.add_argument("--purge", action="store_true")
            p.add_argument("--fp32", action="store_true")
            p.add_argument("--n_layers", type=int, default=12)
            p.add_argument("--n_heads", type=int, default=12)
    args = ap.parse_args()
    {"run": cmd_run, "analyze": cmd_analyze}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
# 4. Environment check
import torch, transformers, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "Enable a GPU accelerator in Session options."
print(torch.cuda.get_device_name(0))

In [ ]:
# 5. Optional Hugging Face token (Add-ons -> Secrets, label HF_TOKEN)
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret; downloading anonymously")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

In [ ]:
# 6. Ablate predictive / random heads in the 30 held-out test models (resumable)
!python feather_causal.py run --purge 2>&1 | grep -v "Loading weights"

In [ ]:
# 7. Summary
!python feather_causal.py analyze

In [ ]:
# 8. Package for download (Output tab)
!zip -qr causal_out.zip feather_out/causal feather_out/causal_report.txt && ls -lh causal_out.zip